# NeuroLens

**An interactive neuroscience playground built on TRIBE v2**

Explore how the brain responds to video, audio, and text — and discover which AI models think most like humans.

Three modules:
1. **PREDICT** — See predicted brain activation for a stimulus
2. **MATCH** — Find content that triggers specific brain states
3. **EVAL** — Benchmark AI models against biological brain responses

In [ ]:
# NeuroLens v1.0 — Interactive Neuroscience Playground
import sys, os
from pathlib import Path

# Clone repo and set up environment (Colab)
if not os.path.exists('neurolens'):
    if not os.path.exists('tribev2'):
        !git clone https://github.com/facebookresearch/tribev2.git
    os.chdir('tribev2')

sys.path.insert(0, os.getcwd())

# Install dependencies
!pip install -q numpy scipy av ffmpeg-python \
    exca neuralset neuraltrain einops pyyaml huggingface_hub \
    gtts langdetect spacy soundfile Levenshtein julius transformers x_transformers \
    nilearn matplotlib pydantic tqdm plotly ipywidgets 2>/dev/null
!pip install -q -e '.[plotting]' --no-deps 2>/dev/null || true
!pip install -q 'moviepy>=2.1' --force-reinstall 2>/dev/null

# Patch moviepy v2 import for neuralset compatibility
import moviepy
try:
    from moviepy import VideoFileClip
except ImportError:
    from moviepy.video.io.VideoFileClip import VideoFileClip
    moviepy.VideoFileClip = VideoFileClip
    sys.modules['moviepy'].VideoFileClip = VideoFileClip

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import json as _json
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Generate demo cache if no real cache exists
CACHE_DIR = Path('neurolens_cache')
if not (CACHE_DIR / 'stimuli' / 'metadata.json').exists():
    print('No cache found. Generating demo cache with synthetic data...')
    for d in ['stimuli', 'brain_preds', 'roi_summaries']:
        (CACHE_DIR / d).mkdir(parents=True, exist_ok=True)
    for model in ['clip', 'whisper', 'gpt2']:
        (CACHE_DIR / 'embeddings' / model).mkdir(parents=True, exist_ok=True)

    demo_stimuli = [
        {'id': 'demo_001', 'name': 'Nature scene', 'category': 'Silence + Visuals', 'media_type': 'video', 'duration_sec': 10.0},
        {'id': 'demo_002', 'name': 'Speech excerpt', 'category': 'Speech', 'media_type': 'video', 'duration_sec': 12.0},
        {'id': 'demo_003', 'name': 'Classical music', 'category': 'Music', 'media_type': 'audio', 'duration_sec': 15.0},
        {'id': 'demo_004', 'name': 'Horror clip', 'category': 'Emotional', 'media_type': 'video', 'duration_sec': 8.0},
        {'id': 'demo_005', 'name': 'Movie scene', 'category': 'Multimodal-rich', 'media_type': 'video', 'duration_sec': 10.0},
    ]
    (CACHE_DIR / 'stimuli' / 'metadata.json').write_text(_json.dumps(demo_stimuli, indent=2))

    np.random.seed(42)
    roi_names = ['Visual Cortex', 'Auditory Cortex', 'Language Areas', 'Motor Cortex',
                 'Prefrontal Cortex', 'Temporal Cortex', 'Parietal Cortex',
                 'Somatosensory Cortex', 'Face-Selective Areas']
    for s in demo_stimuli:
        sid = s['id']
        preds = np.random.randn(8, 20484).astype(np.float32)
        np.savez(CACHE_DIR / 'brain_preds' / f'{sid}.npz', preds=preds)
        roi = {name: round(float(np.random.rand()), 3) for name in roi_names}
        (CACHE_DIR / 'roi_summaries' / f'{sid}.json').write_text(_json.dumps(roi))
        for model in ['clip', 'whisper', 'gpt2']:
            torch.save(torch.randn(256), CACHE_DIR / 'embeddings' / model / f'{sid}.pt')
    print(f'Demo cache ready: {len(demo_stimuli)} stimuli, 3 models')
    print('Note: Using synthetic data. Run neurolens_precompute.ipynb for real brain predictions.\n')

from neurolens.cache import CacheManager
from neurolens.stimulus import StimulusLibrary
from neurolens.predict import get_prediction_at_time, get_num_timesteps, get_top_rois
from neurolens.match import find_similar_stimuli, build_target_from_regions, find_contrast_stimuli
from neurolens.eval import compute_all_model_alignments, compute_model_brain_alignment
from neurolens.roi import get_roi_group_names, ROI_GROUPS
from neurolens.viz import plot_brain_surface, make_radar_chart

cache = CacheManager(CACHE_DIR)
library = StimulusLibrary(CACHE_DIR)
print(f'Loaded {len(library)} stimuli, {len(cache.available_models())} models')

---
## Module 1: PREDICT
Select a stimulus and explore how it activates different brain regions.

In [ ]:
# Stimulus picker
stim_dropdown = widgets.Dropdown(
    options=library.dropdown_options(),
    description="Stimulus:",
    style={"description_width": "initial"},
)

# Time slider (updated dynamically)
time_slider = widgets.IntSlider(
    value=0, min=0, max=1, step=1,
    description="Timestep:",
    continuous_update=False,
)

# View selector
view_select = widgets.SelectMultiple(
    options=["left", "right", "medial_left", "medial_right", "dorsal"],
    value=["left", "right"],
    description="Views:",
)

output_predict = widgets.Output()

def update_predict(*args):
    sid = stim_dropdown.value
    n_steps = get_num_timesteps(cache, sid)
    time_slider.max = n_steps - 1

    with output_predict:
        clear_output(wait=True)
        data = get_prediction_at_time(cache, sid, time_slider.value)
        stim = library.get(sid)
        fig = plot_brain_surface(
            data,
            views=list(view_select.value),
            title=f"{stim.name} (t={time_slider.value})",
        )
        plt.show()

        # Top ROIs
        top = get_top_rois(cache, sid, k=5)
        print("\nTop activated regions:")
        for name, val in top:
            bar = "\u2588" * int(abs(val) * 20)
            print(f"  {name:.<30s} {val:+.3f} {bar}")

stim_dropdown.observe(update_predict, names="value")
time_slider.observe(update_predict, names="value")
view_select.observe(update_predict, names="value")

display(widgets.VBox([
    widgets.HBox([stim_dropdown, time_slider]),
    view_select,
    output_predict,
]))
update_predict()

---
## Module 2: MATCH
Find content that activates specific brain regions, or discover neurally similar stimuli.

In [ ]:
# Tab 1: Region picker
match_mode = widgets.ToggleButtons(
    options=["Region Picker", "More Like This", "Contrast"],
    description="Mode:",
)

# Region picker controls
region_dropdowns = {}
for name in get_roi_group_names():
    region_dropdowns[name] = widgets.FloatSlider(
        value=0.0, min=0.0, max=1.0, step=0.1,
        description=name, style={"description_width": "initial"},
        layout=widgets.Layout(width="400px"),
    )
region_box = widgets.VBox(list(region_dropdowns.values()))

# More Like This controls
source_dropdown = widgets.Dropdown(
    options=library.dropdown_options(),
    description="Source:",
    style={"description_width": "initial"},
)

# Contrast controls
max_roi = widgets.Dropdown(options=get_roi_group_names(), description="Maximize:")
min_roi = widgets.Dropdown(options=get_roi_group_names(), description="Minimize:", value=get_roi_group_names()[1])

output_match = widgets.Output()

def run_match(btn=None):
    with output_match:
        clear_output(wait=True)
        ids = library.ids()
        mode = match_mode.value

        if mode == "Region Picker":
            intensities = {name: slider.value for name, slider in region_dropdowns.items()}
            target = build_target_from_regions(intensities)
            results = find_similar_stimuli(cache, target, ids, top_k=5)
        elif mode == "More Like This":
            source_preds = cache.load_brain_preds(source_dropdown.value)
            target = source_preds.mean(axis=0)
            results = find_similar_stimuli(cache, target, ids, top_k=5)
        else:  # Contrast
            results = find_contrast_stimuli(
                cache, ids, max_roi.value, min_roi.value, top_k=5
            )

        print(f"Top matches ({mode}):\n")
        radar_data = {}
        for rank, (sid, score) in enumerate(results, 1):
            stim = library.get(sid)
            print(f"  {rank}. {stim.name} [{stim.category}] \u2014 score: {score:.3f}")
            roi = cache.load_roi_summary(sid)
            if roi and rank <= 3:
                radar_data[stim.name] = roi

        if radar_data:
            fig = make_radar_chart(radar_data, title="ROI Activation Profiles")
            plt.show()

match_btn = widgets.Button(description="Find Matches", button_style="primary")
match_btn.on_click(run_match)

display(widgets.VBox([
    match_mode,
    region_box,
    widgets.HBox([source_dropdown]),
    widgets.HBox([max_roi, min_roi]),
    match_btn,
    output_match,
]))

---
## Module 3: EVAL
Which AI model thinks most like a human brain?
Compare model representations against predicted brain responses.

In [ ]:
output_eval = widgets.Output()

def run_eval(btn=None):
    with output_eval:
        clear_output(wait=True)
        ids = library.ids()
        print("Computing brain alignment scores (RSA)...\n")

        scores = compute_all_model_alignments(cache, ids)

        # Leaderboard
        print("=" * 50)
        print(f"{'Rank':<6}{'Model':<20}{'Brain Alignment':>15}")
        print("=" * 50)
        for rank, (model, score) in enumerate(scores.items(), 1):
            bar = "\u2588" * int(max(0, score) * 30)
            print(f"{rank:<6}{model:<20}{score:>+.4f}  {bar}")
        print("=" * 50)

        if len(scores) >= 2:
            print("\nSelect two models to compare:")

model_a = widgets.Dropdown(
    options=cache.available_models(),
    description="Model A:",
)
model_b = widgets.Dropdown(
    options=cache.available_models(),
    description="Model B:",
    value=cache.available_models()[-1] if len(cache.available_models()) > 1 else cache.available_models()[0],
)

output_compare = widgets.Output()

def compare_models(btn=None):
    with output_compare:
        clear_output(wait=True)
        ids = library.ids()
        print(f"Comparing {model_a.value} vs {model_b.value}...\n")
        score_a = compute_model_brain_alignment(cache, model_a.value, ids)
        score_b = compute_model_brain_alignment(cache, model_b.value, ids)

        print(f"  {model_a.value}: RSA = {score_a:+.4f}")
        print(f"  {model_b.value}: RSA = {score_b:+.4f}")

        for model_name in [model_a.value, model_b.value]:
            print(f"\n--- Brain Report Card: {model_name} ---")
            score = compute_model_brain_alignment(cache, model_name, ids)
            pct = max(0, score) * 100
            print(f"  Overall brain alignment: {pct:.1f}%")

compare_btn = widgets.Button(description="Compare Models", button_style="info")
compare_btn.on_click(compare_models)

eval_btn = widgets.Button(description="Run Leaderboard", button_style="primary")
eval_btn.on_click(run_eval)

display(widgets.VBox([
    eval_btn,
    output_eval,
    widgets.HBox([model_a, model_b, compare_btn]),
    output_compare,
]))

---
## Explore Further

- **TRIBE v2 Paper:** [A Foundation Model of Vision, Audition, and Language for In-Silico Neuroscience](https://ai.meta.com/research/publications/a-foundation-model-of-vision-audition-and-language-for-in-silico-neuroscience/)
- **TRIBE v2 Demo:** [aidemos.atmeta.com/tribev2](https://aidemos.atmeta.com/tribev2/)
- **Add more stimuli:** Edit `neurolens_precompute.ipynb` and re-run
- **Add more models:** Extract embeddings from any HuggingFace model into the cache

Built with NeuroLens | TRIBE v2 (Meta AI)

---

# Complete Results Showcase

All results below were generated automatically using `neurolens/generate_all_results.py` across all 6 stimuli, 3 AI models, and 72 ROI contrast pairs. This section provides a comprehensive, non-interactive view of every module's output for analysis and reporting.

In [ ]:
# === SHOWCASE SETUP ===
import json
from pathlib import Path
from IPython.display import display, Image as IPImage, HTML, Markdown
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

RESULTS_DIR = Path('neurolens_results')
if not RESULTS_DIR.exists():
    print('Run `python -m neurolens.generate_all_results` first to generate results.')
    print('See docs/GETTING_STARTED.md for instructions.')
CACHE_DIR = Path('neurolens_cache')

# Load metadata
stimuli_meta = json.loads((CACHE_DIR / 'stimuli' / 'metadata.json').read_text())
stim_lookup = {s['id']: s for s in stimuli_meta}

def show_images_grid(image_paths, titles=None, cols=5, figsize_per=(3.5, 3)):
    """Display a grid of images from file paths."""
    rows = (len(image_paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(figsize_per[0]*cols, figsize_per[1]*rows))
    if rows == 1 and cols == 1:
        axes = np.array([axes])
    axes = np.array(axes).flatten()
    for i, ax in enumerate(axes):
        if i < len(image_paths) and image_paths[i].exists():
            img = mpimg.imread(str(image_paths[i]))
            ax.imshow(img)
            if titles and i < len(titles):
                ax.set_title(titles[i], fontsize=9, fontweight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print(f"Results directory: {RESULTS_DIR}")
print(f"Stimuli: {len(stimuli_meta)} | Models: {', '.join(json.loads((RESULTS_DIR / 'summary.json').read_text())['models'])}")

## 4.1 PREDICT Results — Brain Activation Across All Stimuli

### Brain Surface Plots: First Timestep (t=0), All 5 Views per Stimulus

In [ ]:
# Show t=0 brain plots for ALL stimuli, all 5 views
views = ['left', 'right', 'medial_left', 'medial_right', 'dorsal']

for stim in stimuli_meta:
    sid = stim['id']
    stim_dir = RESULTS_DIR / 'predict' / sid
    
    # Get key frames from available files
    brain_files = sorted(stim_dir.glob('brain_t*_left.png'))
    timesteps = sorted(set(int(f.stem.split('_')[1][1:]) for f in brain_files))
    
    print(f"\n{'='*80}")
    print(f"  {stim['name']} [{stim['category']}] — {stim['duration_sec']}s, {len(timesteps)} key frames")
    print(f"{'='*80}")
    
    # Show first timestep, all views
    t0 = timesteps[0]
    paths = [stim_dir / f'brain_t{t0:02d}_{v}.png' for v in views]
    titles = [f't={t0} {v}' for v in views]
    show_images_grid(paths, titles, cols=5, figsize_per=(4, 3))
    
    # ROI summary
    roi = json.loads((stim_dir / 'roi_summary.json').read_text())
    sorted_roi = sorted(roi.items(), key=lambda x: x[1], reverse=True)
    print(f"\n  Top ROIs (time-averaged):")
    for name, val in sorted_roi[:5]:
        bar = '█' * int(abs(val) * 40)
        sign = '+' if val >= 0 else '-'
        print(f"    {name:<25s} {sign}{abs(val):.3f}  {bar}")

### Temporal Dynamics — How Brain Activation Evolves Over Time

Key frames (first, middle, last) for each stimulus, left hemisphere view:

In [ ]:
# Temporal evolution: left view across all key frames for each stimulus
for stim in stimuli_meta:
    sid = stim['id']
    stim_dir = RESULTS_DIR / 'predict' / sid
    brain_files = sorted(stim_dir.glob('brain_t*_left.png'))
    timesteps = sorted(set(int(f.stem.split('_')[1][1:]) for f in brain_files))
    
    paths = [stim_dir / f'brain_t{t:02d}_left.png' for t in timesteps]
    titles = [f't={t} ({"first" if i==0 else "middle" if i==1 else "last"})' 
              for i, t in enumerate(timesteps)]
    
    print(f"\n{stim['name']} [{stim['category']}] — temporal evolution ({len(timesteps)} frames)")
    show_images_grid(paths, titles, cols=len(timesteps), figsize_per=(5, 3.5))

### ROI Activation Comparison — All Stimuli Side by Side

In [ ]:
# Comparative ROI heatmap across all stimuli
import matplotlib.pyplot as plt
import numpy as np

roi_names = list(json.loads((RESULTS_DIR / 'predict' / 'clip_001' / 'roi_summary.json').read_text()).keys())
stim_names = [s['name'] for s in stimuli_meta]
stim_ids = [s['id'] for s in stimuli_meta]

# Build matrix
matrix = np.zeros((len(stim_ids), len(roi_names)))
for i, sid in enumerate(stim_ids):
    roi = json.loads((RESULTS_DIR / 'predict' / sid / 'roi_summary.json').read_text())
    for j, rn in enumerate(roi_names):
        matrix[i, j] = roi.get(rn, 0.0)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(matrix, cmap='RdBu_r', aspect='auto', vmin=-0.45, vmax=0.45)
ax.set_xticks(range(len(roi_names)))
ax.set_xticklabels(roi_names, rotation=45, ha='right', fontsize=10)
ax.set_yticks(range(len(stim_names)))
categories = [s['category'] for s in stimuli_meta]
ax.set_yticklabels([f"{n} [{c}]" for n, c in zip(stim_names, categories)], fontsize=10)
plt.colorbar(im, ax=ax, label='Mean Activation', shrink=0.8)
ax.set_title('ROI Activation Heatmap — All Stimuli', fontsize=14, fontweight='bold')

# Annotate cells
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        color = 'white' if abs(val) > 0.2 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)

plt.tight_layout()
plt.show()

# Category-level summary
print("\nCategory-level ROI averages:")
for cat in sorted(set(categories)):
    cat_indices = [i for i, c in enumerate(categories) if c == cat]
    cat_avg = matrix[cat_indices].mean(axis=0)
    print(f"\n  {cat}:")
    for j, rn in enumerate(roi_names):
        bar = '█' * int(abs(cat_avg[j]) * 30)
        sign = '+' if cat_avg[j] >= 0 else '-'
        print(f"    {rn:<25s} {sign}{abs(cat_avg[j]):.3f}  {bar}")

---

## 4.2 MATCH Results — Neural Similarity and Contrast Analysis

### "More Like This" — Neural Neighbors for Every Stimulus

In [ ]:
# More Like This — show matches + radar charts for all stimuli
mlt_dir = RESULTS_DIR / 'match' / 'more_like_this'

# Neural similarity matrix
sim_matrix = np.zeros((len(stim_ids), len(stim_ids)))
for i, sid_a in enumerate(stim_ids):
    matches = json.loads((mlt_dir / f'{sid_a}_matches.json').read_text())
    match_dict = {m['stimulus_id']: m['similarity'] for m in matches}
    for j, sid_b in enumerate(stim_ids):
        sim_matrix[i, j] = match_dict.get(sid_b, 0.0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Similarity matrix
ax = axes[0]
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(len(stim_names)))
ax.set_xticklabels(stim_names, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(stim_names)))
ax.set_yticklabels(stim_names, fontsize=9)
plt.colorbar(im, ax=ax, label='Cosine Similarity', shrink=0.8)
ax.set_title('Neural Similarity Matrix', fontsize=12, fontweight='bold')
for i in range(len(stim_ids)):
    for j in range(len(stim_ids)):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=8,
                color='white' if sim_matrix[i,j] > 0.5 else 'black')

# Top matches summary
ax = axes[1]
ax.axis('off')
text_lines = ["NEURAL NEIGHBORS (Top Match for Each)\n"]
for sid in stim_ids:
    matches = json.loads((mlt_dir / f'{sid}_matches.json').read_text())
    source_name = stim_lookup[sid]['name']
    # Skip self-match (similarity=1.0)
    best = [m for m in matches if m['stimulus_id'] != sid][0]
    text_lines.append(f"{source_name} → {best['name']} (sim={best['similarity']:.3f})")
ax.text(0.05, 0.95, '\n'.join(text_lines), transform=ax.transAxes, fontsize=11,
        verticalalignment='top', fontfamily='monospace')

plt.tight_layout()
plt.show()

# Show radar charts
print("\nRadar charts — ROI profiles of top 3 neural neighbors:")
radar_paths = sorted(mlt_dir.glob('*_radar.png'))
radar_titles = [stim_lookup[p.stem.replace('_radar', '')]['name'] for p in radar_paths]
show_images_grid(radar_paths, radar_titles, cols=3, figsize_per=(5, 4.5))

### Contrast Analysis — Key ROI Pairs and Directional Asymmetries

In [ ]:
# Key contrast pairs with directional asymmetry analysis
contrast_dir = RESULTS_DIR / 'match' / 'contrast'

key_contrasts = [
    ('Visual_Cortex', 'Auditory_Cortex'),
    ('Auditory_Cortex', 'Visual_Cortex'),
    ('Language_Areas', 'Visual_Cortex'),
    ('Visual_Cortex', 'Language_Areas'),
    ('Motor_Cortex', 'Language_Areas'),
    ('Auditory_Cortex', 'Language_Areas'),
    ('Prefrontal_Cortex', 'Visual_Cortex'),
    ('Face-Selective_Areas', 'Auditory_Cortex'),
]

print("KEY CONTRAST RESULTS — Top stimulus for each directed ROI pair\n")
print(f"{'Maximize':<25s} {'Minimize':<25s} {'Winner':<25s} {'Score':>8s}")
print("=" * 88)

for max_roi, min_roi in key_contrasts:
    fname = f'max_{max_roi}_min_{min_roi}.json'
    fpath = contrast_dir / fname
    if fpath.exists():
        matches = json.loads(fpath.read_text())
        top = matches[0]
        print(f"{max_roi.replace('_', ' '):<25s} {min_roi.replace('_', ' '):<25s} "
              f"{top['name']:<25s} {top['contrast_score']:>+.4f}")

# Directional asymmetry analysis
print(f"\n\nDIRECTIONAL ASYMMETRIES — Same ROI pair, opposite directions\n")
asymmetry_pairs = [
    ('Visual_Cortex', 'Auditory_Cortex'),
    ('Language_Areas', 'Visual_Cortex'),
    ('Motor_Cortex', 'Language_Areas'),
    ('Prefrontal_Cortex', 'Temporal_Cortex'),
]

for a, b in asymmetry_pairs:
    fwd = json.loads((contrast_dir / f'max_{a}_min_{b}.json').read_text())
    rev = json.loads((contrast_dir / f'max_{b}_min_{a}.json').read_text())
    
    print(f"\n  {a.replace('_', ' ')} vs {b.replace('_', ' ')}:")
    print(f"    Maximize {a.replace('_', ' ')}: #{1} {fwd[0]['name']} ({fwd[0]['contrast_score']:+.4f})")
    print(f"    Maximize {b.replace('_', ' ')}: #{1} {rev[0]['name']} ({rev[0]['contrast_score']:+.4f})")
    if fwd[0]['stimulus_id'] != rev[0]['stimulus_id']:
        print(f"    → ASYMMETRIC: different winners!")
    else:
        print(f"    → Symmetric: same stimulus wins both directions")

# Show radar charts for selected contrasts
print("\n\nRadar charts for key contrasts:")
selected_radars = [
    contrast_dir / 'max_Visual_Cortex_min_Auditory_Cortex_radar.png',
    contrast_dir / 'max_Auditory_Cortex_min_Visual_Cortex_radar.png',
    contrast_dir / 'max_Language_Areas_min_Visual_Cortex_radar.png',
    contrast_dir / 'max_Motor_Cortex_min_Language_Areas_radar.png',
]
titles = ['Visual > Auditory', 'Auditory > Visual', 'Language > Visual', 'Motor > Language']
show_images_grid(selected_radars, titles, cols=2, figsize_per=(6, 5))

---

## 4.3 EVAL Results — AI Model vs Brain Alignment

### Model Leaderboard and Pairwise Comparisons

In [ ]:
# Leaderboard
eval_dir = RESULTS_DIR / 'eval'
leaderboard = json.loads((eval_dir / 'leaderboard.json').read_text())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax = axes[0]
models = [e['model'].upper() for e in leaderboard]
scores = [e['rsa_score'] for e in leaderboard]
colors = ['#2ecc71' if s > 0.1 else '#f39c12' if s > 0 else '#e74c3c' for s in scores]
bars = ax.bar(models, scores, color=colors, edgecolor='black', linewidth=0.5, width=0.5)
ax.set_ylabel('RSA Score (Spearman r)', fontsize=12)
ax.set_title('Brain Alignment Leaderboard', fontsize=14, fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{score:+.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylim(-0.25, 0.55)

# Pairwise comparison table
ax = axes[1]
ax.axis('off')

compare_files = sorted(eval_dir.glob('compare_*.json'))
lines = ["PAIRWISE MODEL COMPARISONS\n"]
for cf in compare_files:
    c = json.loads(cf.read_text())
    winner_marker_a = " ◀ WINNER" if c['winner'] == c['model_a'] else ""
    winner_marker_b = " ◀ WINNER" if c['winner'] == c['model_b'] else ""
    lines.append(f"{c['model_a'].upper():>8s}: RSA = {c['rsa_a']:+.4f}  ({c['alignment_pct_a']:.1f}%){winner_marker_a}")
    lines.append(f"{c['model_b'].upper():>8s}: RSA = {c['rsa_b']:+.4f}  ({c['alignment_pct_b']:.1f}%){winner_marker_b}")
    lines.append(f"{'Gap':>8s}: {abs(c['rsa_a'] - c['rsa_b']):.4f}")
    lines.append("")

ax.text(0.05, 0.95, '\n'.join(lines), transform=ax.transAxes, fontsize=11,
        verticalalignment='top', fontfamily='monospace')

plt.tight_layout()
plt.show()

# Summary
print("\n" + "=" * 60)
print("SUMMARY: Which AI model thinks most like a brain?")
print("=" * 60)
for e in leaderboard:
    pct = e['brain_alignment_pct']
    bar = '█' * int(pct)
    print(f"  #{e['rank']} {e['model'].upper():<10s} RSA={e['rsa_score']:+.4f}  "
          f"Alignment={pct:.1f}%  {bar}")
print(f"\nCLIP dominates — visual representations best match brain patterns for video stimuli.")

---

## 4.4 Complete All-Contrasts Gallery

All 72 directed ROI contrast radar charts, organized by maximize ROI:

In [ ]:
# Gallery of all contrast radar charts grouped by maximize ROI
contrast_dir = RESULTS_DIR / 'match' / 'contrast'
roi_names_sorted = sorted(set(
    f.stem.split('_min_')[0].replace('max_', '').replace('_', ' ')
    for f in contrast_dir.glob('max_*_radar.png')
))

for max_roi in roi_names_sorted:
    safe_roi = max_roi.replace(' ', '_')
    radars = sorted(contrast_dir.glob(f'max_{safe_roi}_min_*_radar.png'))
    if not radars:
        continue
    titles = [f.stem.replace(f'max_{safe_roi}_min_', '').replace('_radar', '').replace('_', ' ')
              for f in radars]
    print(f"\n{'─'*60}")
    print(f"  Maximize: {max_roi} — contrasted against {len(radars)} other ROIs")
    print(f"{'─'*60}")
    show_images_grid(radars, titles, cols=4, figsize_per=(4.5, 4))

---

# 5. Findings Report

## Key Discovery: Visual Dominance in Brain Responses to Video

Across all three modules, **visual processing emerges as the dominant factor** in how the brain represents video stimuli:

| Evidence | Module | Finding |
|----------|--------|---------|
| Visual Cortex modulation | Predict | Widest activation range across categories (+0.169 to -0.161) |
| Visual similarity clusters | Match | Stimuli cluster by visual content more than by audio |
| CLIP wins brain alignment | Eval | RSA = 0.386, far ahead of Whisper (0.029) and GPT-2 (-0.143) |

## Top 7 Findings

### 1. Speech Stimuli Form the Tightest Neural Cluster (sim=0.801)
Jack Ma and Ronaldo motivational speeches share 80.1% neural similarity -- the highest pair in the dataset. Both activate Auditory Cortex (+0.21 avg) and Motor Cortex (+0.15 avg), consistent with the motor theory of speech perception. However, they diverge internally: Jack Ma is linguistically dominant (Language Areas +0.129), Ronaldo is motor-dominant (+0.207) -- speech *style* matters as much as speech *presence*.

### 2. Music is the Strongest Language Suppressor (-0.414)
High Impact Music drives Language Areas to **-0.414** -- the single most extreme activation in either direction across all stimuli and ROIs. This suppression *intensifies* over time (colorbar doubles from -1.0 to -2.0 between t=0 and t=8). Music also paradoxically *suppresses* Auditory Cortex (-0.175), suggesting it engages motor/rhythmic circuits rather than classical auditory pathways.

### 3. CLIP Massively Outperforms Audio and Text Models (13x)
Single-frame CLIP embeddings align with brain representations 13x better than full-audio Whisper. GPT-2 actually **anti-correlates** (RSA = -0.143), meaning its text-based similarity structure actively contradicts the brain's perceptual organization. An RSA of 0.386 is in the range neuroscience papers report for well-matched IT cortex models (Kriegeskorte et al., 2008).

### 4. Same-Category Stimuli Can Be Neurally Orthogonal (sim=0.001)
Muay Thai Kick and Romantic Couple are both "Silence + Visuals" but have near-zero neural similarity (0.001). Muay Thai is visuospatially rich (Parietal +0.152, Face-Selective +0.131), while Romantic Couple is globally suppressed (max ROI = +0.034). **Content category labels don't map to neural response patterns.**

### 5. Motor-Language Is the Most Discriminative Brain Axis (12x asymmetry)
The Motor vs Language contrast produces the widest spread (0.731) and most extreme directional asymmetry: maximize Motor/minimize Language yields +0.491 (Music wins), but the reverse yields only +0.041 (Jack Ma wins). A **12x asymmetry** -- the Motor-over-Language pole is far more neurally separable than Language-over-Motor.

### 6. Motor Cortex Is the Most Universal Brain Response
Motor Cortex is the **only ROI with a positive grand mean** (+0.084) across all 6 stimuli. It appears in the top 3 ROIs for 4/6 stimuli. Embodied motor simulation is the most universal brain response to video, regardless of content type.

### 7. Temporal Dynamics Reveal Progressive Engagement
Brain activation follows an **inverted-U arc** for speech (peak at mid-clip t=5), but shows **progressive intensification** for music (doubling suppression by t=8). Medial views reveal cingulate cortex engagement (emotion/attention) and precuneus activation during music (default mode network / internal imagery).

## Category Prediction from Neural Patterns

| Category | Prediction Rule | Accuracy |
|----------|----------------|----------|
| Speech | Auditory > 0.1 AND Visual < 0 | **100%** |
| Music | Language < -0.3 | **100%** |
| Silence + Visuals | Parietal > 0.15 AND Face-Selective > 0.1 | **67%** (fails Romantic Couple) |

## Limitations
- Only 6 stimuli (minimum for RSA, statistically underpowered; need 20+)
- CLIP uses single middle frame, not full video
- GPT-2 uses stimulus name, not actual transcription
- No permutation-based significance testing or confidence intervals
- Simulated brain data (TRIBE v2 predictions, not actual fMRI/EEG)
- Coarse 9-region ROIs average over fine-grained cortical areas

## Recommended Next Steps
1. Expand to 20+ stimuli for robust RSA statistics
2. Add permutation p-values (10,000x label shuffles)
3. Add video-native models (VideoMAE, InternVideo)
4. Compute region-specific RSA (CLIP vs visual cortex only)
5. Use actual speech transcriptions for GPT-2
6. Add subcortical ROIs (amygdala, striatum) to explain Romantic Couple's flat cortical profile

*Full report with complete quantitative analysis: `neurolens_results/NEUROLENS_FINDINGS_REPORT.md`*

---
*NeuroLens | Built on TRIBE v2 (Meta AI)*